# Vignette 3: Image-conditioned augmentation

The paper's strongest generation methods started from a real image: **inpaint** replaced the
central square of a photograph, **in-then-outpaint** replaced the border afterwards so every
pixel was regenerated while the composition survived. This notebook runs both methods with SDXL
(the only backend with a hard-mask inpainting pipeline in diffusers), then runs the 2026
replacement, **reference-image editing** with Qwen-Image-2.1 ("the same condition on a
different patient"), on the same real images.

It then reproduces the paper's Figure 2B analysis: CLIP embedding distances between synthetic
images and the real image they came from (matched) against random real-synthetic and real-real
pairs, plus a near-duplicate check against the training set.

In [ ]:
SMALL = True
N_REFERENCES = 3 if SMALL else 12
NUM_PER_IMAGE = 2
RESOLUTION = 768
SEED = 0

In [ ]:
import os, json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less fragmentation on a 32 GB card
WORK = Path(os.environ.get("SYNDERM_WORK_DIR", "/workspace/synderm"))
WORK.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def show_grid(rows, row_titles=None, col_titles=None, size=2.6, suptitle=None):
    """rows: list of lists of PIL images (one list per row)."""
    n_rows, n_cols = len(rows), max(len(r) for r in rows)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * n_rows), squeeze=False)
    for r, row in enumerate(rows):
        for c in range(n_cols):
            ax = axes[r][c]; ax.axis("off")
            if c < len(row) and row[c] is not None:
                ax.imshow(row[c])
            if r == 0 and col_titles and c < len(col_titles):
                ax.set_title(col_titles[c], fontsize=10)
        if row_titles:
            axes[r][0].text(-0.08, 0.5, row_titles[r], transform=axes[r][0].transAxes, rotation=90,
                            va="center", ha="right", fontsize=10)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()

In [ ]:
from synderm.data import load_sample
train = load_sample(split="train")
labels = train.labels
# one reference image per label
refs = []
seen = set()
for i in range(len(train)):
    lab = train.df.iloc[i]["label"]
    if lab not in seen:
        seen.add(lab); refs.append(i)
    if len(refs) >= N_REFERENCES: break
from torch.utils.data import Subset
ref_ds = Subset(train, refs)
show_grid([[ref_ds[i]["image"] for i in range(len(ref_ds))]], col_titles=[ref_ds[i]["label"] for i in range(len(ref_ds))], suptitle="Reference images")

## 1. Hard-mask inpainting with SDXL (the paper's methods)

`masks.center_mask` is a white square covering half of each side, as in the paper;
`border_mask` is its complement. `generate_synthetic_dataset(method="inpaint-outpaint")` applies
both in sequence with the paper's strengths (0.97 then 0.95).

In [ ]:
from synderm.backends import get_backend, masks
from synderm.generation import generate_synthetic_dataset
size = ref_ds[0]["image"].size
show_grid([[ref_ds[0]["image"], masks.center_mask(size), masks.border_mask(size)]], col_titles=["reference", "inpaint mask", "outpaint mask"], size=2.4)
sdxl = get_backend("sdxl")
gen_root = WORK / "generations"
sdxl_in = generate_synthetic_dataset(sdxl, gen_root / "sdxl-inpaint", method="inpaint", dataset=ref_ds, num_per_image=NUM_PER_IMAGE, seed=SEED)
sdxl_io = generate_synthetic_dataset(sdxl, gen_root / "sdxl-inpaint-outpaint", method="inpaint-outpaint", dataset=ref_ds, num_per_image=NUM_PER_IMAGE, seed=SEED)
sdxl.unload()

## 2. Reference-image editing with Qwen-Image-2.1

The edit prompt asks for the same disease on a different patient and body site. The model sees
the reference through its vision encoder and as latent tokens; nothing is masked.

In [ ]:
qwen = get_backend("qwen-image-2.1")
print("edit prompt:", qwen.edit_caption(ref_ds[0]["label"]))
qwen_edit = generate_synthetic_dataset(qwen, gen_root / "qwen-edit", method="edit", dataset=ref_ds, num_per_image=NUM_PER_IMAGE,
                                       height=RESOLUTION, width=RESOLUTION, seed=SEED)
qwen.unload()

In [ ]:
def by_ref(ds):
    out = {}
    for i in range(len(ds)):
        it = ds[i]; out.setdefault(it["ref_id"], []).append(it["image"])
    return out
sets = {"SDXL inpaint": by_ref(sdxl_in), "SDXL in-then-outpaint": by_ref(sdxl_io), "Qwen edit": by_ref(qwen_edit)}
rows, titles = [], []
for i in range(len(ref_ds)):
    it = ref_ds[i]
    rows.append([it["image"]] + [sets[k][it["id"]][0] for k in sets])
    titles.append(it["label"])
show_grid(rows, row_titles=titles, col_titles=["reference"] + list(sets), size=2.4)

## 3. Are the generations new images or copies? (paper Figure 2B)

The paper embedded images with CLIP ViT-B/32 and compared cosine distances: synthetic images
were much closer to their own reference (0.08) than random real-synthetic pairs (0.24), which
matched random real-real pairs. The same statistic is computed here for each method. A method
whose matched distance approaches the unmatched distance produces images that are no longer
tied to their source; one whose matched distance is near zero is copying.

In [ ]:
from synderm.eval import ImageEmbedder, embed_dataset, matched_vs_unmatched_distances, nearest_real_neighbors
emb = ImageEmbedder()  # CLIP ViT-B/32 image tower
real_feats, real_meta = embed_dataset(train, emb)
real_ids = [m["id"] if "id" in m else m["md5hash"] for m in real_meta]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
summary = {}
for ax, (name, ds) in zip(axes, {"SDXL inpaint": sdxl_in, "SDXL in-then-outpaint": sdxl_io, "Qwen edit": qwen_edit}.items()):
    f, meta = embed_dataset(ds, emb)
    d = matched_vs_unmatched_distances(real_feats, real_ids, f, [m["ref_id"] for m in meta], n_unmatched=2000)
    for k, v in d.items():
        ax.hist(v, bins=30, alpha=0.5, label=f"{k} (mean {v.mean():.3f})", density=True)
    sim, _ = nearest_real_neighbors(real_feats, f)
    summary[name] = {"matched": d["matched"].mean(), "real_synthetic": d["real_synthetic"].mean(),
                     "real_real": d["real_real"].mean(), "max_sim_to_any_real": sim.max()}
    ax.set_title(name); ax.set_xlabel("cosine distance"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
pd.DataFrame(summary).T.round(3)

`max_sim_to_any_real` above about 0.95 flags a near-duplicate of a training image. With a
handful of references the numbers are only illustrative; vignette 6 runs the same metrics at
dataset scale.

In our runs the Qwen edit stayed closest to its reference (matched distance about 0.09, near
the paper's 0.08 for in-then-outpaint): it preserved skin tone and removed the Atlas watermark
but kept the composition almost unchanged, so "a different patient at a different body site" is
only weakly followed at the default (guidance-free) sampling. Two ways to push diversity:
pass `guidance_scale=4.0` (classifier-free guidance with the edit prompt) or generate from text
with a LoRA (vignette 2) and keep editing for cases where the composition should survive.